In [1]:
import inspect

import sys
from pathlib import Path

root = r"D:\Projects\speech-ai-learning"

if root not in sys.path:
    sys.path.insert(0, str(root))

from src import asr_inference
from src import metrics
from src import normalize

print("========== ASR Inference ==========")
print(inspect.getsource(asr_inference))

print("\n========== Metrics ==========")
print(inspect.signature(metrics.score_tokens))
print(inspect.signature(metrics.score_corpus))

print("\n========== Normalize ==========")
print(inspect.signature(normalize.normalize_score))
print(inspect.signature(normalize.mixed_tokens))

========== ASR Inference ==========
from pathlib import Path

import torch

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
)

from src.audio_utils import (
    load_audio,
    to_mono,
    resample_audio,
)

def load_whisper_model(
        checkpoint="openai/whisper-small",
        device=None
):
    """
    加载 Whisper Processor 和模型。

    返回：
        processor
        model
        device
    """

    #自动选择设备
    if device is None:
        device = ("cuda" if torch.cuda.is_available() else "cpu")

    device = torch.device(device)

    #加载Processor
    processor = WhisperProcessor.from_pretrained(checkpoint)

    #加载模型
    model = WhisperForConditionalGeneration.from_pretrained(checkpoint)

    model = model.to(device)

    #切换评估模式
    model.eval()

    print(f"Checkpoint: {checkpoint}")
    print(f"Device: {device}")

    return processor, model, device

def transcribe_file(
        audio_path,
        processor,
        model,
        device=Non

In [2]:
#检查Metrics的实际实现及评分版本
import inspect
from src import metrics
from src import normalize

print("========== score_tokens ==========")
print(inspect.getsource(metrics.score_tokens))

print("\n========== score_corpus ==========")
print(inspect.getsource(metrics.score_corpus))

print("NORMALIZER_VERSION:")
print(getattr(normalize, "NORMALIZER_VERSION", "未定义"))

========== score_tokens ==========
def score_tokens(reference_tokens, hypothesis_tokens):
    """
    计算 token 级编辑距离、S/D/I/C 和错误率。
    """
    dp = build_dp(reference_tokens,hypothesis_tokens)

    operations = backtrace(
        reference_tokens,
        hypothesis_tokens,
        dp
    )

    counts = {
        "S": 0,
        "D": 0,
        "I": 0,
        "C": 0
    }

    for op, _, _ in operations:
        counts[op] += 1

    distance = dp[-1][-1]
    N = len(reference_tokens)

    assert (
        counts["S"] + counts["D"] + counts["I"]
        == distance
    )

    #空reference的特殊处理在模块3继续讨论
    error_rate = distance / N if N > 0 else None

    return {
        "S": counts["S"],
        "D": counts["D"],
        "I": counts["I"],
        "C": counts["C"],
        "N": N,
        "distance": distance,
        "error_rate": error_rate,
        "operations": operations,
    }


========== score_corpus ==========
def score_corpus(token_pairs):
    """
    计算语料级错误率。

    token_pai

In [3]:
#确认normalize_score() 和 mixed_tokens() 的实际实现
import inspect
from src import normalize

print("========== normalize_score ==========")
print(inspect.getsource(normalize.normalize_score))

print("\n========== mixed_tokens ==========")
print(inspect.getsource(normalize.mixed_tokens))

========== normalize_score ==========
def normalize_score(text: str) -> str:
    """
    将原始文本转换为统一评分文本。

    规则：
    1. NFKC
    2. 英文小写
    3. 普通标点转为空格
    4. 非数字间的句点转为空格
    5. 折叠空白
    6. 保留未知有效符号
    """
    if not isinstance(text, str):
        raise TypeError("text must be a string")

    #1. Unicode兼容规范化
    text = unicodedata.normalize("NFKC", text)

    #2.英文小写(lower也会处理其他有大小写的文字)
    text = text.lower()

    #对不可安全解释的控制字符显式报错
    for char in text:
        category = unicodedata.category(char)

        if (
            category in {"Cc", "Cf", "Cs"}
            and not char.isspace()
        ):
            raise ValueError(
                f"Unsupported control character: U+{ord(char):04X}"
            )

    #3.普通标点替换为空格
    for punctuation in ORDINARY_PUNCTUATION:
        text = text.replace(punctuation, " ")

    #4. 数字之间的'.'保留：
    #    3.14、v1.2.3 保留
    #    hello.world -> hello world
    text = re.sub(
        r"(?<!\d)\.|\.(?!\d)",
        " ",
        text
    )

   

In [4]:
from src.evaluate import cer_tokens, mer_tokens

text = "我在2026年学习Python"

print("CER:", cer_tokens(text))
print("CER数量:", len(cer_tokens(text)))

print("\nMER:", mer_tokens(text))
print("MER数量:", len(mer_tokens(text)))

assert len(cer_tokens(text)) == 15
assert len(mer_tokens(text)) == 7

print("\nTokenization验收通过")

CER: ['我', '在', '2', '0', '2', '6', '年', '学', '习', 'p', 'y', 't', 'h', 'o', 'n']
CER数量: 15

MER: ['我', '在', '2026', '年', '学', '习', 'python']
MER数量: 7

Tokenization验收通过


In [5]:
#运行验收加载评估数据
import sys
from pathlib import Path

root = r"D:\Projects\speech-ai-learning"

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.evaluate import load_eval_data

csv_path = (
    r"D:\Projects\speech-ai-learning"
    r"\metadata\dev_candidates_W7D5.csv"
)

eval_df = load_eval_data(csv_path)

print("有效评估样本数:", len(eval_df))

print("\n字段:")
print(eval_df.columns.tolist())

print("\n样本信息:")
print(
    eval_df[
        ["uid", "reference", "score_reference"]
    ].head()
)

assert len(eval_df) == 10

assert (
    eval_df["status"] == "verified"
).all()

print("\nDev 数据读取验收通过")

有效评估样本数: 10

字段:
['selection_rank', 'uid', 'audio_path', 'duration_sec', 'speaker_id', 'parent_uid', 'source_text', 'reference', 'score_reference', 'status', 'review_minutes', 'split', 'uncertain_span', 'notes']

样本信息:
                     uid                              reference  \
0  cv26_3f66c228bc2a4ae8                        这两家公司都在网上出售树莓派。   
1  cv26_1753be3b6083f28f                                 武藏国出身。   
2  cv26_20558b6a5c8dbac9                      另一联赛是阿根廷全国足球甲级联赛。   
3  cv26_40123f9681ffefa4                  措布伦是奥地利蒂罗尔州罗伊特县的一个市镇。   
4  cv26_8e8d4f65babf0b7e  维济马农村居民点是俄罗斯联邦沃洛格达州别洛泽尔斯克区所属的一个农村居民点。   

                        score_reference  
0                        这两家公司都在网上出售树莓派  
1                                 武藏国出身  
2                      另一联赛是阿根廷全国足球甲级联赛  
3                  措布伦是奥地利蒂罗尔州罗伊特县的一个市镇  
4  维济马农村居民点是俄罗斯联邦沃洛格达州别洛泽尔斯克区所属的一个农村居民点  

Dev 数据读取验收通过


In [11]:
#Notebook中运行10条语音推理
import sys
from pathlib import Path

root = r"D:\Projects\speech-ai-learning"

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.evaluate import (
    load_eval_data,
    run_batch_inference
)

from src.asr_inference import load_whisper_model

#1. 读取Dev数据
csv_path = (
    r"D:\Projects\speech-ai-learning"
    r"\metadata\dev_candidates_W7D5.csv"
)

eval_df = load_eval_data(csv_path)

# 确认全部来自Dev
assert (eval_df["split"] == "dev").all()

#2. 加载模型(只加载一次)
processor, model, device = load_whisper_model(
    checkpoint="openai/whisper-small"
)

#3.批量推理
predictions = run_batch_inference(
    eval_df=eval_df,
    processor=processor,
    model=model,
    device=device,
    language="zh",
    task="transcribe",
    max_new_tokens=96
)

#4. 查看结果
print("\n预测结果：")

print(
    predictions[
        [
            "uid",
            "reference",
            "hypothesis"
        ]
    ].to_string(index=False)
)

print("\n批量推理样本数:", len(predictions))

assert len(predictions) == 10

print("\nBatch Inference 验收通过")

2026-11-07 10:24:08,797 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/revision/main "HTTP/1.1 200 OK"
2026-11-07 10:24:09,522 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-07 10:24:09,866 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-07 10:24:10,201 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-11-07 10:24:10,653 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/revision/main "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

2026-11-07 10:24:11,841 - INFO - Loaded D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22057623.mp3,shape=torch.Size([1, 285696]), sr=48000
[transformers] Both `max_new_tokens` (=96) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Checkpoint: openai/whisper-small
Device: cuda
[1/10] Transcribing cv26_3f66c228bc2a4ae8


2026-11-07 10:24:12,096 - INFO - Loaded D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004281.mp3,shape=torch.Size([1, 133632]), sr=48000
[transformers] Both `max_new_tokens` (=96) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-11-07 10:24:12,244 - INFO - Loaded D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004016.mp3,shape=torch.Size([1, 269568]), sr=48000
[transformers] Both `max_new_tokens` (=96) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/10] Transcribing cv26_1753be3b6083f28f
[3/10] Transcribing cv26_20558b6a5c8dbac9


... (10 outputs left out)

2026-11-07 10:24:14,538 - INFO - Loaded D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22212051.mp3,shape=torch.Size([1, 407808]), sr=48000
[transformers] Both `max_new_tokens` (=96) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[9/10] Transcribing cv26_1441a8cec768fc20


2026-11-07 10:24:14,885 - INFO - Loaded D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004245.mp3,shape=torch.Size([1, 133632]), sr=48000
[transformers] Both `max_new_tokens` (=96) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[10/10] Transcribing cv26_dc1278883fe4eb06

预测结果：
                  uid                             reference                           hypothesis
cv26_3f66c228bc2a4ae8                       这两家公司都在网上出售树莓派。                       這兩家公司都在網上出售數沒派
cv26_1753be3b6083f28f                                武藏国出身。                                五藏國出身
cv26_20558b6a5c8dbac9                     另一联赛是阿根廷全国足球甲级联赛。                     另一连赛是阿根廷全国足球假籍连赛
cv26_40123f9681ffefa4                 措布伦是奥地利蒂罗尔州罗伊特县的一个市镇。              错不论是奥地利、蒂罗尔州、罗伊特县的一个市镇。
cv26_8e8d4f65babf0b7e 维济马农村居民点是俄罗斯联邦沃洛格达州别洛泽尔斯克区所属的一个农村居民点。 維吉馬農村居民點是俄羅斯聯邦沃爾格達州別洛澤爾斯克區所屬的一個農村居民點
cv26_dc3e083eab508fd5                              编组共两节车厢。                              边组供两节车厢
cv26_f05564f22bc1d77c          德国空军将西班牙内战学习的经验第一次运用在大规模战争上。          德國空軍將西班牙內戰學習的經驗第一次運用在大規模戰爭上
cv26_2bd1b28412804c60              在接下来的几个月里，“信天翁”在旧金山境外运营。                在接下來的幾個月裡,信天翁在舊金山境外運營
cv26_1441a8cec768fc20                  阿韦利诺洛佩斯是巴西皮奥伊州的一个市镇。                 阿

In [12]:
#计算评分
import sys
from pathlib import Path

root = r"D:\Projects\speech-ai-learning"

if root not in sys.path:
    sys.path.insert(0, str(root))
from src.evaluate import compute_corpus_metrics

predictions_with_metrics, corpus_metrics = (
    compute_corpus_metrics(predictions)
)

print("========== Corpus Metrics ==========")

for key, value in corpus_metrics.items():
    print(f"{key}: {value}")

print("\n========== Sample Metrics ==========")

print(
    predictions_with_metrics[
        [
            "uid",
            "cer_error_rate",
            "mer_error_rate"
        ]
    ].to_string(index=False)
)

assert len(predictions_with_metrics) == 10

assert corpus_metrics["CER"] is not None
assert corpus_metrics["MER"] is not None

print("\nCorpus Metrics 验收通过")

========== Corpus Metrics ==========
num_samples: 10
CER: 0.33136094674556216
CER_S: 54
CER_D: 2
CER_I: 0
CER_N: 169
MER: 0.33136094674556216
MER_S: 54
MER_D: 2
MER_I: 0
MER_N: 169

========== Sample Metrics ==========
                  uid  cer_error_rate  mer_error_rate
cv26_3f66c228bc2a4ae8        0.357143        0.357143
cv26_1753be3b6083f28f        0.400000        0.400000
cv26_20558b6a5c8dbac9        0.250000        0.250000
cv26_40123f9681ffefa4        0.150000        0.150000
cv26_8e8d4f65babf0b7e        0.472222        0.472222
cv26_dc3e083eab508fd5        0.285714        0.285714
cv26_f05564f22bc1d77c        0.481481        0.481481
cv26_2bd1b28412804c60        0.409091        0.409091
cv26_1441a8cec768fc20        0.000000        0.000000
cv26_dc1278883fe4eb06        0.333333        0.333333

Corpus Metrics 验收通过


In [13]:
#输出结果并保存配置
from pathlib import Path

import importlib
import src.evaluate 

importlib.reload(src.evaluate)

from src.evaluate import save_evaluation_results

output_dir = (
    Path(r"D:\Projects\speech-ai-learning")
    / "results"
    / "w07d6"
    / "whisper_small_baseline"
)

csv_path, json_path = save_evaluation_results(
    predictions=predictions_with_metrics,
    corpus_metrics=corpus_metrics,
    model=model,
    output_dir=output_dir,
    checkpoint="openai/whisper-small",
    split="dev",
    language="zh",
    task="transcribe",
    max_new_tokens=96
)

assert csv_path.exists()
assert json_path.exists()

print("\n结果保存验收通过")

2026-11-07 10:24:25,224 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small "HTTP/1.1 200 OK"


实验结果保存成功
CSV: D:\Projects\speech-ai-learning\results\w07d6\whisper_small_baseline\predictions.csv
JSON: D:\Projects\speech-ai-learning\results\w07d6\whisper_small_baseline\metadata.json

结果保存验收通过


In [14]:
#检验输出的metadata
import json
import pandas as pd

csv_path = (
    r"D:\Projects\speech-ai-learning"
    r"\results\w07d6\whisper_small_baseline"
    r"\predictions.csv"
)

json_path = (
    r"D:\Projects\speech-ai-learning"
    r"\results\w07d6\whisper_small_baseline"
    r"\metadata.json"
)

predictions_check = pd.read_csv(csv_path)

with open(json_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("========== Metadata ==========")

for key, value in metadata.items():
    print(f"{key}: {value}")

print("\n========== CSV Check ==========")

print("样本数:", len(predictions_check))
print("UID唯一:", predictions_check["uid"].is_unique)
print("字段:", predictions_check.columns.tolist())

assert len(predictions_check) == 10
assert predictions_check["uid"].is_unique
assert metadata["split"] == "dev"
assert metadata["num_samples"] == 10
assert metadata["model_revision"] is not None

print("\n最终保存检查通过")

========== Metadata ==========
created_at: 2026-11-07T16:24:25.441027+00:00
split: dev
model_name: openai/whisper-small
model_revision: 973afd24965f72e36ca33b3055d56a652f456b4d
language: zh
task: transcribe
max_new_tokens: 96
device: cuda:0
normalizer_version: w7d4-v1-draft
scoring_version: w7d6-v1
num_samples: 10
metrics: {'num_samples': 10, 'CER': 0.33136094674556216, 'CER_S': 54, 'CER_D': 2, 'CER_I': 0, 'CER_N': 169, 'MER': 0.33136094674556216, 'MER_S': 54, 'MER_D': 2, 'MER_I': 0, 'MER_N': 169}

========== CSV Check ==========
样本数: 10
UID唯一: True
字段: ['split', 'uid', 'audio_path', 'reference', 'hypothesis', 'score_reference', 'score_hypothesis', 'duration_sec', 'cer_S', 'cer_D', 'cer_I', 'cer_C', 'cer_N', 'cer_distance', 'cer_error_rate', 'mer_S', 'mer_D', 'mer_I', 'mer_C', 'mer_N', 'mer_distance', 'mer_error_rate']

最终保存检查通过
